<a href="https://colab.research.google.com/github/SoniaElizabeth/ICT_DSA_2026/blob/main/Data_AcquisitionCase_Study.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import requests
import random

###Step 1: Load Launch Data

In [21]:
launch_url= "https://api.spacexdata.com/v4/launches"
launch_data= requests.get(launch_url).json()

In [22]:
launch_df= pd.DataFrame(launch_data)

In [23]:
launch_df= launch_df[['name', 'date_utc', 'success', 'details', 'rocket']]

In [24]:
launch_df['date_utc']= pd.to_datetime(launch_df['date_utc'])
launch_df['year']= launch_df['date_utc'].dt.year

###Step 2: Load Rocket Data

In [25]:
rocket_url= "https://api.spacexdata.com/v4/rockets"
rocket_data= requests.get(rocket_url).json()

rocket_df= pd.DataFrame(rocket_data)

rocket_df= rocket_df[['id', 'name', 'type', 'active', 'stages']]
rocket_df.rename(columns={'id': 'rocket'}, inplace=True)

###Step 3: Merge Data

In [26]:
merged_df= pd.merge(launch_df, rocket_df, on='rocket', how='left')

###Step 4: Add Simulated Country

In [27]:
countries= ['USA', 'Russia', 'India', 'China', 'France']
merged_df['country']= [random.choice(countries) for _ in range(len(merged_df))]
print(merged_df.head())

        name_x                  date_utc success  \
0    FalconSat 2006-03-24 22:30:00+00:00   False   
1      DemoSat 2007-03-21 01:10:00+00:00   False   
2  Trailblazer 2008-08-03 03:34:00+00:00   False   
3       RatSat 2008-09-28 23:15:00+00:00    True   
4     RazakSat 2009-07-13 03:35:00+00:00    True   

                                             details  \
0   Engine failure at 33 seconds and loss of vehicle   
1  Successful first stage burn and transition to ...   
2  Residual stage 1 thrust led to collision betwe...   
3  Ratsat was carried to orbit on the first succe...   
4                                               None   

                     rocket  year    name_y    type  active  stages country  
0  5e9d0d95eda69955f709d1eb  2006  Falcon 1  rocket   False       2  France  
1  5e9d0d95eda69955f709d1eb  2007  Falcon 1  rocket   False       2  France  
2  5e9d0d95eda69955f709d1eb  2008  Falcon 1  rocket   False       2   China  
3  5e9d0d95eda69955f709d1eb  2008  Fal

###Step 5: Store in SQL Lite 3

In [28]:
import sqlite3
conn= sqlite3.connect("spacex.db")
merged_df.to_sql("launches", conn, if_exists="replace", index=False)
print("Data stored successfully")

Data stored successfully


##SQL Queries

###1. Launches by Country

In [29]:
query1= """ SELECT country, COUNT(*) AS total_launches
FROM launches
GROUP BY country
ORDER BY total_launches DESC; """
print(pd.read_sql(query1, conn))

  country  total_launches
0   China              49
1     USA              46
2   India              45
3  France              37
4  Russia              28


###2. Year with Highest Launches

In [30]:
query2= """ SELECT year, COUNT(*) AS total_launches
FROM launches
GROUP BY year
ORDER BY total_launches DESC
LIMIT 1; """
print(pd.read_sql(query2, conn))

   year  total_launches
0  2022              62


###3. Top 5 missions by Launch count

In [31]:
query3= """ SELECT name_x, COUNT(*) AS launch_count
FROM launches
GROUP BY name_x
ORDER BY launch_count DESC
LIMIT 5; """
print(pd.read_sql(query3, conn))

                      name_x  launch_count
0  ispace Mission 1 & Rashid             1
1                       ZUMA             1
2     WorldView Legion 1 & 2             1
3        Viasat-3 & Arcturus             1
4                    USSF-44             1
